# Pairwise rank manifest stats

This notebook reads the pairwise rank manifests from `data/manifests/score_prediction_rank/` and shows clip pair-count min/mean/max for `all`, `train`, `val`, and `test`.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from aitraf.datasets.score_prediction_rank import ScorePredictionRankDataset

PROJECT_ROOT = Path("..").resolve()
MANIFESTS_DIR = PROJECT_ROOT / "data" / "manifests" / "score_prediction_rank"

In [ ]:
split_dfs = {
    split: pd.DataFrame(
        ScorePredictionRankDataset(
            manifests_dir=MANIFESTS_DIR, split=split
        ).manifest_rows()
    )
    for split in ["train", "val", "test"]
}

rows = []

for split, df in [
    ("all", pd.concat(split_dfs.values(), ignore_index=True)),
    *split_dfs.items(),
]:
    clip_counts = (
        pd.concat([df["left_s3_path"], df["right_s3_path"]])
        .value_counts()
        .rename_axis("clip")
        .rename("pair_count")
    )
    rows.append(
        {
            "split": split,
            "pairs": len(df),
            "clips": int(clip_counts.shape[0]),
            "min_pair_count": int(clip_counts.min()),
            "mean_pair_count": float(clip_counts.mean()),
            "max_pair_count": int(clip_counts.max()),
        }
    )

display(pd.DataFrame(rows))